# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

### Method Choice for Lane 2: Refresh / Opportunity Scoring
To solve our operational ranking problem ("Which 20–50 URLs should the editorial team review first?"), we train and compare three distinct modeling paradigms against our Week 4 Heuristic Baseline:

1. **Linear Baseline — Logistic Regression (with L2 Regularization & Standard Scaling):**
   - Establishes the performance ceiling of linear decision boundaries.
   - Provides an easily interpretable baseline where feature coefficients directly indicate directional risk.

2. **Transparent Tree — Decision Tree Classifier (max_depth=5, min_samples_leaf=50):**
   - Serves as a readable, human-interpretable set of hierarchical decision rules.
   - Captures non-linear thresholds (e.g., distinguishing Page 1 positions from striking distance and deep tail).

3. **Core Ensemble — Random Forest Classifier (100 trees, max_depth=10, min_samples_leaf=25):**
   - Combines multiple decorrelated decision trees using bootstrap aggregation and random feature sub-sampling.
   - Captures tangled non-linear interactions across search volume (`impressions_90d`), SERP vulnerability (`avg_position`), search presence consistency (`days_with_impressions`), and content depth (`word_count`) while resisting overfitting.
   - Outputs smooth, well-calibrated posterior probabilities $P(\text{decline} \mid X)$ to power a finely graded priority ranking queue.

### Why Complexity Must Earn Its Keep
We do not adopt Random Forest simply because it is an ensemble; it must demonstrably beat simpler linear and single-tree models on **Precision@K** on held-out domains to justify its operational deployment.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust path resolution for local, repo root, and Google Colab
data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
eligible["is_declining"] = (eligible["trend_direction"] == "down").astype(int)

# Feature matrix construction: strictly pre-decision observable signals
num_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = [
    "competition_level", "content_type", "main_intent", "age_tier", 
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

X_num = eligible[num_cols].fillna(0)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume"]:
    X_num["log_" + c] = np.log1p(X_num[c])

X_cat = pd.get_dummies(eligible[cat_cols].fillna("unknown").astype(str), prefix=cat_cols, dtype=float)
X_all = pd.concat([X_num, X_cat], axis=1)
y = eligible["is_declining"].values

print(f"Dataset Slice: {len(eligible):,} rows across {eligible['client_id'].nunique()} clients.")
print(f"Feature Matrix Shape: {X_all.shape[0]:,} samples x {X_all.shape[1]} engineered features.")
print(f"Target Distribution: {eligible['is_declining'].mean():.1%} declining (Positive Class).")



Dataset Slice: 30,000 rows across 32 clients.
Feature Matrix Shape: 30,000 samples x 60 engineered features.
Target Distribution: 54.2% declining (Positive Class).


## 2. Split design

### Client-Grouped Holdout Split
In multi-tenant search intelligence, rows belonging to the same client share domain authority, backlink profiles, CMS architecture, industry seasonality, and content creation practices. A naive random row split allows URLs from the same client to appear in both training and test sets, causing severe cross-domain leakage and yielding wildly over-optimistic validation metrics.

To ensure an honest evaluation of real-world generalization, we implement a **Client-Holdout Split**:
- **Unit of Grouping:** `client_id` (pseudonymized domain cluster).
- **Split Ratio:** 80% train clients, 20% test clients (using fixed `RANDOM_STATE = 42`).
- **Mutual Exclusivity:** Zero overlap of client IDs between training and test sets.
- **Evaluation Purpose:** The test set represents entirely unseen client portfolios, proving whether learned decay patterns generalize to new sites.


In [2]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
clients = eligible["client_id"].unique()
shuffled_clients = rng.permutation(clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

train_idx = ~eligible["client_id"].isin(test_clients)
test_idx = eligible["client_id"].isin(test_clients)

X_train, y_train = X_all[train_idx], y[train_idx]
X_test, y_test = X_all[test_idx], y[test_idx]
test_eligible = eligible[test_idx].copy().reset_index(drop=True)

print("=== Split Design Audit (Client-Holdout) ===")
print(f"Train Set: {X_train.shape[0]:,} rows from {eligible[train_idx]['client_id'].nunique()} clients (decline rate: {y_train.mean():.1%})")
print(f"Test Set:  {X_test.shape[0]:,} rows from {len(test_clients)} unseen clients (decline rate: {y_test.mean():.1%})")
print(f"Client Overlap: {len(set(eligible[train_idx]['client_id']).intersection(test_clients))} clients shared (Strictly 0 -> No leakage!)")



=== Split Design Audit (Client-Holdout) ===
Train Set: 27,675 rows from 26 clients (decline rate: 55.5%)
Test Set:  2,325 rows from 6 unseen clients (decline rate: 39.1%)
Client Overlap: 0 clients shared (Strictly 0 -> No leakage!)


## 3. Train + compare vs my baseline

### Fair Comparison on the Exact Same Test Data and Split
Every model is trained strictly on `X_train` and evaluated alongside our Week 4 Heuristic Baseline on the exact same 2,325 test URLs from held-out clients.

### Evaluation Metrics
- **Precision@20 & Precision@50:** Primary business metrics representing the precision of the top review queue cohorts delivered to the editorial team.
- **ROC-AUC & Average Precision (PR-AUC):** Global ranking discrimination across all thresholds.
- **Recall & F1 Score:** Binary classification performance at standard 0.5 decision threshold.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score
import json

# Define Precision@K metric
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

# Percentile rank helper for Week 4 Baseline Rule
def percentile_rank(series):
    return series.rank(pct=True)

# 1. Week 4 Baseline Rule evaluated on TEST set
vis_test = percentile_rank(np.log1p(test_eligible["impressions_90d"]))
pos_norm_test = (1 - (test_eligible["avg_position"].clip(1, 50) - 1) / 49.0) * (test_eligible["avg_position"] > 0).astype(int)
stale_test = percentile_rank(test_eligible["days_since_last_update"])
baseline_test_scores = (0.40 * vis_test + 0.35 * pos_norm_test + 0.25 * stale_test) * 100.0

# 2. Candidate Models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()), 
        ("lr", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "Random Forest (Capstone Model)": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, 
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    )
}

comparison_rows = []
# Baseline Row
comparison_rows.append({
    "Method / Model": "Baseline Rules (Week 4)",
    "ROC-AUC": float(roc_auc_score(y_test, baseline_test_scores)),
    "Avg Precision": float(average_precision_score(y_test, baseline_test_scores)),
    "Precision@20": precision_at_k(baseline_test_scores, y_test, 20),
    "Precision@50": precision_at_k(baseline_test_scores, y_test, 50),
    "Recall": np.nan,
    "F1": np.nan
})

# Fit models and evaluate
model_probs = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    probs = m.predict_proba(X_test)[:, 1]
    model_probs[name] = probs
    preds = (probs >= 0.5).astype(int)
    comparison_rows.append({
        "Method / Model": name,
        "ROC-AUC": float(roc_auc_score(y_test, probs)),
        "Avg Precision": float(average_precision_score(y_test, probs)),
        "Precision@20": precision_at_k(probs, y_test, 20),
        "Precision@50": precision_at_k(probs, y_test, 50),
        "Recall": float(recall_score(y_test, preds)),
        "F1": float(f1_score(y_test, preds))
    })

comparison_df = pd.DataFrame(comparison_rows)
print("=== Final Model Comparison on Client-Holdout Test Set (n=2,325) ===")
print(comparison_df.round(3).to_string(index=False))
print(f"\nBase Decline Rate on Test Set: {y_test.mean():.1%}")

# Save comparison receipts
out_dir = Path("work/outputs") if Path("work/outputs").exists() else Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
receipt_path = out_dir / "model_comparison_metrics.json"

receipt_data = {
    "assignment": "ML-08",
    "test_samples": len(y_test),
    "test_clients": len(test_clients),
    "base_rate": float(y_test.mean()),
    "comparison": comparison_rows
}
with open(receipt_path, "w") as f:
    json.dump(receipt_data, f, indent=2)
print(f"Saved model comparison receipt to: {receipt_path}")



=== Final Model Comparison on Client-Holdout Test Set (n=2,325) ===
                Method / Model  ROC-AUC  Avg Precision  Precision@20  Precision@50  Recall    F1
       Baseline Rules (Week 4)    0.622          0.468          0.45          0.34     NaN   NaN
           Logistic Regression    0.705          0.525          0.40          0.36   0.551 0.563
                 Decision Tree    0.742          0.575          0.50          0.56   0.716 0.634
Random Forest (Capstone Model)    0.754          0.632          0.90          0.76   0.741 0.643

Base Decline Rate on Test Set: 39.1%
Saved model comparison receipt to: ../outputs/model_comparison_metrics.json


## 4. Errors and interpretation

### Feature Importance Interpretation
The Random Forest model distributes weight across multiple observable signal dimensions:
1. `days_with_impressions` (~12.4%): Measures how consistently the search engine surfaces the page. Pages with sporadic, intermittent impression days exhibit high volatility.
2. `avg_position` (~10.1%): Captures ranking tier vulnerability. Pages on Page 1 and striking distance face maximum decay risk.
3. `impressions_90d` / `log_impressions_90d` (~17.5% combined): Search exposure volume dictates the magnitude of potential traffic loss.
4. `content_age_days` (~7.0%): Measures catalog maturity.
5. `word_count` / `char_count` (~7.4% combined): Thin content relative to query intent signals susceptibility to competitive displacement.

*Sanity Check:* No single feature dominates with >15% importance, confirming that the model learns balanced multivariate interactions rather than leaking an individual column.

---

### Concrete Error Analysis (Inspecting Where the Model Breaks)

1. **False Positives (High Predicted Probability, Actual Trend: Not Down):**
   - *Example (`content_db1cd41b4b4f`):* Model probability = 0.734, but actual trend was `up`.
   - *Diagnosis:* The URL had an average position of 12.9 (striking distance), high staleness (105 days), and moderate volume (1,482 impressions)—matching the typical decaying profile. However, it gained momentum due to external keyword expansion that internal content metrics cannot see.

2. **False Negatives (Low Predicted Probability, Actual Trend: Down):**
   - *Example (`content_28b4223f4e5f`):* Model probability = 0.057, but actual trend was `down`.
   - *Diagnosis:* The page had only 1 impression and average position of 0.0 (no ranking data). A page with 1 impression dropping to 0 is technically classified as `down` in percentage terms, but has virtually no demand signal for the model to detect. This proves why minimum volume thresholds ($\ge 100$ impressions) are essential in production to filter out noise.


In [4]:
# Feature importance analysis
rf_model = models["Random Forest (Capstone Model)"]
importances = pd.Series(rf_model.feature_importances_, index=X_all.columns).sort_values(ascending=False)

print("=== Top 10 Feature Importances (Random Forest) ===")
print(importances.head(10).round(4).to_string())

# Concrete error cases
rf_probs = model_probs["Random Forest (Capstone Model)"]
test_eligible["rf_prob"] = rf_probs
test_eligible["rf_pred"] = (rf_probs >= 0.5).astype(int)

# 1. False Positives (predicted decay, but actually stable/up)
fps = test_eligible[(test_eligible["rf_pred"] == 1) & (test_eligible["is_declining"] == 0)].sort_values("rf_prob", ascending=False)
print("\n=== Top 3 False Positive Error Cases (High Score, Actual Stable/Up) ===")
print(fps[["content_id", "client_id", "rf_prob", "impressions_90d", "avg_position", "days_since_last_update", "trend_direction"]].head(3).to_string(index=False))

# 2. False Negatives (predicted safe, but actually down)
fns = test_eligible[(test_eligible["rf_pred"] == 0) & (test_eligible["is_declining"] == 1)].sort_values("rf_prob", ascending=True)
print("\n=== Top 3 False Negative Error Cases (Low Score, Actual Down) ===")
print(fns[["content_id", "client_id", "rf_prob", "impressions_90d", "avg_position", "days_since_last_update", "trend_direction"]].head(3).to_string(index=False))



=== Top 10 Feature Importances (Random Forest) ===
days_with_impressions    0.1236
avg_position             0.1008
impressions_90d          0.0909
log_impressions_90d      0.0848
content_age_days         0.0701
age_tier_365+            0.0396
word_count               0.0373
char_count               0.0364
log_clicks_90d           0.0249
clicks_90d               0.0244

=== Top 3 False Positive Error Cases (High Score, Actual Stable/Up) ===
          content_id         client_id  rf_prob  impressions_90d  avg_position  days_since_last_update trend_direction
content_db1cd41b4b4f client_f74efabef1 0.733838             1482          12.9                     105              up
content_643f585dc7f7 client_f74efabef1 0.726920              761          25.1                      20              up
content_b15a8dbdf66f client_f74efabef1 0.726319             1647          22.4                      20              up

=== Top 3 False Negative Error Cases (Low Score, Actual Down) ===
          con

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
